In [42]:
import numpy as np
from matplotlib import pyplot as plt
import math
import scipy.constants as constants
import numba
get_ipython().run_line_magic('matplotlib', 'auto')

Using matplotlib backend: MacOSX


In [47]:
plt.rcParams.update({'font.size': 17}) # keep those graph fonts readable!
plt.rcParams['figure.dpi'] = 120

def init(xmax):
    plt.xlim((0, xmax-1))
    plt.grid('on')
    ax.set_xlabel('Grid Cells ($z$)')
    ax.set_ylabel('$E_z (eV^2)$')
    plt.show()

def graph(t, E):
    plt.clf()
    ax = fig.add_axes([.25, .25, .6, .6])
    
    img = ax.contourf(E)
    draw_circle = plt.Circle((jsource, isource), int(sourceRadius), fill=False)
    ax.add_artist(draw_circle)
    cbar=plt.colorbar(img, ax=ax)
    cbar.set_label('$E_{phy,z}$ (eV^2)')
    ax.set_title('frame time{}'.format(t))
    plt.show()
    plt.pause(0.01)

## Pulse (trivial)

In [25]:
# stability requirements
dx = 20e-9 # ~ nm grid size
dt_si = dx*0.5/constants.c # ~3.3e-17 s in one grid

# pulse 
dt_fs = dt_si/constants.femto # ! ~0.033 fs in one grid
spread = 2/dt_fs # 1/df_fs = num of grids in 1 fs, spread = num of grids in 2 fs ~ 60 grids
t0 = spread*4 # offset by 60*6=360 grids
freq_in = 2*math.pi* 50* constants.tera # angular frequency = 1.256e15 rad s^-1
w_scale = freq_in*dt_si # 0.042 rad per grid

# set time in grid unit
nsteps = 1000
t = np.arange(0,nsteps+1)

@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*w_scale)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.show()
print(t0, spread)

239.83396640000004 59.95849160000001


In [26]:
"""
For stability reason, I impose two criteria:
1) dt/dx = 0.5
2) dt2 = 0.25
Therefore dx = sqrt(dt2)/0.25 = 1; dt = 0.5
"""

@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion):
    Ex_phy = (Ex + kappa*axion*Bx)/(1+kappa**2*axion**2)
    Ey_phy = (Ey + kappa*axion*By)/(1+kappa**2*axion**2)
    Ez_phy = (Ez + kappa*axion*Bz)/(1+kappa**2*axion**2)
    Bx_phy = (Bx - kappa*axion*Ex)/(1+kappa**2*axion**2)
    By_phy = (By - kappa*axion*Ey)/(1+kappa**2*axion**2)
    Bz_phy = (Bz - kappa*axion*Ez)/(1+kappa**2*axion**2)
    
    return Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion):
    Ex = Ex_phy - kappa*axion*Bx_phy
    Ey = Ey_phy - kappa*axion*By_phy
    Ez = Ez_phy - kappa*axion*Bz_phy
    Bx = Bx_phy + kappa*axion*Ex_phy
    By = By_phy + kappa*axion*Ey_phy
    Bz = Bz_phy + kappa*axion*Ez_phy
    
    return Ex, Ey, Ez, Bx, By, Bz

@numba.jit(nopython=True)
def Eupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(1,kmax-1):
        for i in range(1,kmax-1):
            Ex[j,i] = Ex[j,i] + 0.5*(Bz[j,i] - Bz[j-1,i])
            Ey[j,i] = Ey[j,i] + 0.5*(Bz[j-1,i] - Bz[j,i])
            Ez[j,i] = Ez[j,i] + 0.5*(By[j,i]-By[j,i-1]+Bx[j-1,i]-Bx[j,i])
    return Ex, Ey, Ez

@numba.jit(nopython=True)
def Bupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(0,kmax-1):
        for i in range(0,kmax-1):
            Bx[j,i] = Bx[j,i] + 0.5*(Ez[j,i]-Ez[j+1,i])
            By[j,i] = By[j,i] + 0.5*(Ez[j,i+1]-Ez[j,i])
            Bz[j,i] = Bz[j,i] + 0.5*(Ex[j+1,i] - Ex[j,i]+Ey[j,i]-Ey[j,i+1])
    return Bx, By, Bz

@numba.jit(nopython=True)
def Aupdate2d(Ex, Ey, Ez, Bx, By, Bz, axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy):
    for j in range(1, kmax-1):
        for i in range(1,kmax-1):
            axion[j,i] = 2*axion[j,i] - axion_past[j,i] + 0.25*(axion[j,i+1]+axion[j+1,i]-4*axion[j,i]\
            +axion[j,i-1]+axion[j-1,i]) - dt2*kappa* (Ex_phy[j,i]*Bx_phy[j,i] + Ey_phy[j,i]*By_phy[j,i]\
            +Ez_phy[j,i]*Bz_phy[j,i]) - dt2*m**2 * axion[j,i]
        
    return axion



In [27]:
# axion params
m = 1
kappa =1 

# stability
dt2 = 0.25

# initialization
kmax = 1000
Ex = Ey = Ez = np.zeros([kmax,kmax])
Ex_phy = Ey_phy = Ez_phy = np.zeros([kmax,kmax])
Bx = By = Bz = np.zeros([kmax,kmax])
Bx_phy = By_phy = Bz_phy = np.zeros([kmax,kmax])
axion = np.zeros([kmax,kmax])

# source
isource = int(kmax/2)
jsource = int(kmax/2)
nsteps = 1000

# stability requirements
dx = 20e-9 # ~ nm grid size
dt_si = dx*0.5/constants.c # ~3.3e-17 s in one grid

# pulse 
dt_fs = dt_si/constants.femto # ! ~0.033 fs in one grid
spread = 2/dt_fs # 1/df_fs = num of grids in 1 fs, spread = num of grids in 2 fs ~ 60 grids
t0 = spread*4 # offset by 60*6=360 grids
freq_in = 2*math.pi* 50* constants.tera # angular frequency = 1.256e15 rad s^-1
w_scale = freq_in*dt_si # 0.042 rad per grid


@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*w_scale)
    return source

cycle = 100
fig = plt.figure(figsize=(8,6))

for t in range(nsteps+1):
    pulse = get_source(t)
    
    if t == 0:
        axion_past = np.zeros([kmax, kmax])
    
    # update E
    Ex, Ey, Ez = Eupdate2d(Ex, Ey, Ez, Bx, By, Bz)
    # update physical fields
    Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
    # inject E source
    Ez_phy[jsource, isource] = Ez_phy[jsource, isource] + pulse
    # update hat fields
    Ex, Ey, Ez, Bx, By, Bz = phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion)
    
    # update B 
    Bx, By, Bz = Bupdate2d(Ex, Ey, Ez, Bx, By, Bz)
    # update physical fields
    Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
    
    # update A
    axion = Aupdate2d(Ex, Ey, Ez, Bx, By, Bz, axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy)
    
    if t == 0:
        axion_current = np.zeros([kmax,kmax])
    axion_past = axion_current
    axion_current = axion
    
    if t % cycle == 0:
        Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
        graph(t, Ez_phy)
    

## Pulse ($B_{external}$ injected after specific time in all space)

In [44]:
nsteps = 1000
t = np.arange(0,nsteps+1)
spread = 100
t0 = spread*4

@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*np.pi*0.01)
    return source

#plt.xlim(0,20)
plt.clf()
plt.plot(t, get_source(t))
plt.show()
print(t0, spread)

400 100


In [45]:
"""
For stability reason, I impose two criteria:
1) dt/dx = 0.5
2) dt2 = 0.25
Therefore dx = sqrt(dt2)/0.25 = 1; dt = 0.5
"""

dt2 = 0.25
kappa = 1

@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion):
    Ex_phy = (Ex + kappa*axion*Bx)/(1+kappa**2*axion**2)
    Ey_phy = (Ey + kappa*axion*By)/(1+kappa**2*axion**2)
    Ez_phy = (Ez + kappa*axion*Bz)/(1+kappa**2*axion**2)
    Bx_phy = (Bx - kappa*axion*Ex)/(1+kappa**2*axion**2)
    By_phy = (By - kappa*axion*Ey)/(1+kappa**2*axion**2)
    Bz_phy = (Bz - kappa*axion*Ez)/(1+kappa**2*axion**2)
    
    return Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion):
    Ex = Ex_phy - kappa*axion*Bx_phy
    Ey = Ey_phy - kappa*axion*By_phy
    Ez = Ez_phy - kappa*axion*Bz_phy
    Bx = Bx_phy + kappa*axion*Ex_phy
    By = By_phy + kappa*axion*Ey_phy
    Bz = Bz_phy + kappa*axion*Ez_phy
    
    return Ex, Ey, Ez, Bx, By, Bz

@numba.jit(nopython=True)
def Eupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(1,kmax-1):
        for i in range(1,kmax-1):
            Ex[j,i] = Ex[j,i] + 0.5*(Bz[j,i] - Bz[j-1,i])
            Ey[j,i] = Ey[j,i] + 0.5*(Bz[j-1,i] - Bz[j,i])
            Ez[j,i] = Ez[j,i] + 0.5*(By[j,i]-By[j,i-1]+Bx[j-1,i]-Bx[j,i])
    return Ex, Ey, Ez

@numba.jit(nopython=True)
def Bupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(0,kmax-1):
        for i in range(0,kmax-1):
            Bx[j,i] = Bx[j,i] + 0.5*(Ez[j,i]-Ez[j+1,i])
            By[j,i] = By[j,i] + 0.5*(Ez[j,i+1]-Ez[j,i])
            Bz[j,i] = Bz[j,i] + 0.5*(Ex[j+1,i] - Ex[j,i]+Ey[j,i]-Ey[j,i+1])
    return Bx, By, Bz

@numba.jit(nopython=True)
def Aupdate2d(Ex, Ey, Ez, Bx, By, Bz, axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy):
    for j in range(1, kmax-1):
        for i in range(1,kmax-1):
            axion[j,i] = 2*axion[j,i] - axion_past[j,i] + 0.25*(axion[j,i+1]+axion[j+1,i]-4*axion[j,i]\
            +axion[j,i-1]+axion[j-1,i]) - dt2*kappa* (Ex_phy[j,i]*Bx_phy[j,i] + Ey_phy[j,i]*By_phy[j,i]\
            +Ez_phy[j,i]*Bz_phy[j,i]) - dt2*m**2 * axion[j,i]
        
    return axion



In [52]:
"""
For stability reason, I impose two criteria:
1) dt/dx = 0.5
2) dt2 = 0.25
Therefore dx = sqrt(dt2)/0.25 = 1; dt = 0.5
"""

dt2 = 0.25
kappa = 1


@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion):
    Ex_phy = (Ex + kappa*axion*Bx)/(1+kappa**2*axion**2)
    Ey_phy = (Ey + kappa*axion*By)/(1+kappa**2*axion**2)
    Ez_phy = (Ez + kappa*axion*Bz)/(1+kappa**2*axion**2)
    Bx_phy = (Bx - kappa*axion*Ex)/(1+kappa**2*axion**2)
    By_phy = (By - kappa*axion*Ey)/(1+kappa**2*axion**2)
    Bz_phy = (Bz - kappa*axion*Ez)/(1+kappa**2*axion**2)
    
    return Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion):
    Ex = Ex_phy - kappa*axion*Bx_phy
    Ey = Ey_phy - kappa*axion*By_phy
    Ez = Ez_phy - kappa*axion*Bz_phy
    Bx = Bx_phy + kappa*axion*Ex_phy
    By = By_phy + kappa*axion*Ey_phy
    Bz = Bz_phy + kappa*axion*Ez_phy
    
    return Ex, Ey, Ez, Bx, By, Bz

@numba.jit(nopython=True)
def Eupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(1,kmax-1):
        for i in range(1,kmax-1):
            Ex[j,i] = Ex[j,i] + 0.5*(Bz[j,i] - Bz[j-1,i])
            Ey[j,i] = Ey[j,i] + 0.5*(Bz[j-1,i] - Bz[j,i])
            Ez[j,i] = Ez[j,i] + 0.5*(By[j,i]-By[j,i-1]+Bx[j-1,i]-Bx[j,i])
    return Ex, Ey, Ez

@numba.jit(nopython=True)
def Bupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(0,kmax-1):
        for i in range(0,kmax-1):
            Bx[j,i] = Bx[j,i] + 0.5*(Ez[j,i]-Ez[j+1,i])
            By[j,i] = By[j,i] + 0.5*(Ez[j,i+1]-Ez[j,i])
            Bz[j,i] = Bz[j,i] + 0.5*(Ex[j+1,i] - Ex[j,i]+Ey[j,i]-Ey[j,i+1])
    return Bx, By, Bz

@numba.jit(nopython=True)
def Aupdate2d(Ex, Ey, Ez, Bx, By, Bz, axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy):
    for j in range(1, kmax-1):
        for i in range(1,kmax-1):
            axion[j,i] = 2*axion[j,i] - axion_past[j,i] + 0.25*(axion[j,i+1]+axion[j+1,i]-4*axion[j,i]\
            +axion[j,i-1]+axion[j-1,i]) - dt2*kappa* (Ex_phy[j,i]*Bx_phy[j,i] + Ey_phy[j,i]*By_phy[j,i]\
            +Ez_phy[j,i]*Bz_phy[j,i]) - dt2*m**2 * axion[j,i]
        
    return axion


# control
plot1d = True


# axion params
m = 1
kappa =1 

# stability
dt2 = 0.25

# initialization
kmax = 1000
Ex = Ey = Ez = np.zeros([kmax,kmax])
Ex_phy = Ey_phy = Ez_phy = np.zeros([kmax,kmax])
Bx = By = Bz = np.zeros([kmax,kmax])
Bx_phy = By_phy = Bz_phy = np.zeros([kmax,kmax])
axion = np.zeros([kmax,kmax])

# source
isource = int(kmax/2)
jsource = int(kmax/2)
bsource = isource +200
nsteps = 1000
spread=100
t0 = spread*4
@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*np.pi*0.01)
    return source



cycle = 100
if plot1d == False:
    plt.clf()
    fig = plt.figure(figsize=(8,6))

if plot1d == True:
    plt.clf()
    plt.close()
    cycle = 100
    lw=2
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_axes([.18, .18, .7, .7])
    xrange = np.linspace(0,kmax, kmax)
    [im] = ax.plot(xrange,Ez[int(kmax/2),:],linewidth=lw)
    [im2] = ax.plot(xrange,By[int(kmax/2),:],linewidth=lw)
    [im3] = ax.plot(xrange,Bx[int(kmax/2),:],linewidth=lw)
    im.set_color('orange')
    im2.set_color('blue')
    im3.set_color('red')
    init(kmax)
    plt.legend(['Ez', 'By', 'Bx'])
    plt.ylim(-1, 1)



for t in range(nsteps+1):
    pulse = get_source(t)
    
    if t == 0:
        axion_past = np.zeros([kmax, kmax])
    
    # update E
    Ex, Ey, Ez = Eupdate2d(Ex, Ey, Ez, Bx, By, Bz)
    # update physical fields
    Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
    # inject E source
    Ez_phy[:, isource] = Ez_phy[:, isource] + pulse
    # update hat fields
    Ex, Ey, Ez, Bx, By, Bz = phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion)
    
    # update B 
    Bx, By, Bz = Bupdate2d(Ex, Ey, Ez, Bx, By, Bz)
    
    if t == 300:
        # update physical fields
        Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
        # inject B source
        Bz_phy[:,bsource:] = Bz_phy[:,bsource:] + 5e-5*np.ones_like(Bz_phy[:,bsource:])
        # update hat fields
        Ex, Ey, Ez, Bx, By, Bz = phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion)
    
    # update physical fields
    Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
    # update A
    axion = Aupdate2d(Ex, Ey, Ez, Bx, By, Bz, axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy)
    
    if t == 0:
        axion_current = np.zeros([kmax,kmax])
    axion_past = axion_current
    axion_current = axion
    
    if t % cycle == 0:
        Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
        if plot1d == False:
            graph(t, Ez_phy)
        if plot1d == True:
            im.set_ydata(Ez_phy[int(kmax/2),:]) # blue
            ax.set_title("frame time {}".format(t))
            plt.show()
            plt.pause(0.05)
print('done')
    

done


## Pulse ($B_{external}$ beyond specific radius at specific time)

In [41]:
"""
For stability reason, I impose two criteria:
1) dt/dx = 0.4
2) dt2 = 0.16
Therefore dx = dt/0.4 = 1; dt = 0.4
"""

# control
plot1d = True


# axion params
m = 1
kappa =1 

# initialization
kmax = 1000
Ex = Ey = Ez = np.zeros([kmax,kmax])
Ex_phy = Ey_phy = Ez_phy = np.zeros([kmax,kmax])
Bx = By = Bz = np.zeros([kmax,kmax])
Bx_phy = By_phy = Bz_phy = np.zeros([kmax,kmax])
axion = np.zeros([kmax,kmax])

# source
isource = int(kmax/2)
jsource = int(kmax/2)
nsteps = 1000
sourceRadius = 0.4*(kmax/2) # Beyond which apply external B field
Bext = 1e-10 # amplitude of B_external

# stability requirements
dx = 20e-9 # ~ nm grid size
dtdx=0.4 # dt/dx = 0.4
dtdx2=0.16 # (dt/dx)^2
dt_si = dx*dtdx/constants.c # ~3.3e-17 s in one grid
hbar= 6.582e-16
dt = dt_si/hbar
dt2 = dt**2



# pulse 
dt_fs = dt_si/constants.femto # ! ~0.033 fs in one grid
spread = 2/dt_fs # 1/df_fs = num of grids in 1 fs, spread = num of grids in 2 fs ~ 60 grids
t0 = spread*4 # offset by 60*6=360 grids
freq_in = 2*math.pi* 50* constants.tera # angular frequency = 6.28e14 rad s^-1
w_scale = freq_in*dt_si # 0.042 rad per grid




@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*w_scale)
    return source




@numba.jit(nopython=True)
def get_source(t):
    source = -np.exp(-0.5*(t-t0)**2/spread**2)*np.cos(t*np.pi*0.01)
    return source

@numba.jit(nopython=True)
def inj_Bsource(Bz_phy):
    for j in range(kmax):
        for i in range(kmax):
            radius2 = (jsource-j)**2 + (isource-i)**2
            if radius2 >= sourceRadius**2:
                Bz_phy[j,i] = Bz_phy[j,i] + Bext
    return Bz_phy


@numba.jit(nopython=True)
def hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion):
    Ex_phy = (Ex + kappa*axion*Bx)/(1+kappa**2*axion**2)
    Ey_phy = (Ey + kappa*axion*By)/(1+kappa**2*axion**2)
    Ez_phy = (Ez + kappa*axion*Bz)/(1+kappa**2*axion**2)
    Bx_phy = (Bx - kappa*axion*Ex)/(1+kappa**2*axion**2)
    By_phy = (By - kappa*axion*Ey)/(1+kappa**2*axion**2)
    Bz_phy = (Bz - kappa*axion*Ez)/(1+kappa**2*axion**2)
    
    return Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy
@numba.jit(nopython=True)
def phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion):
    Ex = Ex_phy - kappa*axion*Bx_phy
    Ey = Ey_phy - kappa*axion*By_phy
    Ez = Ez_phy - kappa*axion*Bz_phy
    Bx = Bx_phy + kappa*axion*Ex_phy
    By = By_phy + kappa*axion*Ey_phy
    Bz = Bz_phy + kappa*axion*Ez_phy
    
    return Ex, Ey, Ez, Bx, By, Bz

@numba.jit(nopython=True)
def Eupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(1,kmax-1):
        for i in range(1,kmax-1):
            Ex[j,i] = Ex[j,i] + dtdx*(Bz[j,i] - Bz[j-1,i])
            Ey[j,i] = Ey[j,i] + dtdx*(Bz[j-1,i] - Bz[j,i])
            Ez[j,i] = Ez[j,i] + dtdx*(By[j,i]-By[j,i-1]+Bx[j-1,i]-Bx[j,i])
    return Ex, Ey, Ez

@numba.jit(nopython=True)
def Bupdate2d(Ex, Ey, Ez, Bx, By, Bz):
    for j in range(0,kmax-1):
        for i in range(0,kmax-1):
            Bx[j,i] = Bx[j,i] + dtdx*(Ez[j,i]-Ez[j+1,i])
            By[j,i] = By[j,i] + dtdx*(Ez[j,i+1]-Ez[j,i])
            Bz[j,i] = Bz[j,i] + dtdx*(Ex[j+1,i] - Ex[j,i]+Ey[j,i]-Ey[j,i+1])
    return Bx, By, Bz

@numba.jit(nopython=True)
def Aupdate2d(axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy):
    for j in range(1, kmax-1):
        for i in range(1,kmax-1):
            axion[j,i] = 2*axion[j,i] - axion_past[j,i] + dtdx2*(axion[j,i+1]+axion[j+1,i]-4*axion[j,i]\
            +axion[j,i-1]+axion[j-1,i]) - dt2*kappa* (Ex_phy[j,i]*Bx_phy[j,i] + Ey_phy[j,i]*By_phy[j,i]\
            +Ez_phy[j,i]*Bz_phy[j,i]) - dt2*m**2 * axion[j,i]
        
    return axion



cycle = 100
if plot1d == False:
    plt.clf()
    fig = plt.figure(figsize=(8,6))

if plot1d == True:
    plt.clf()
    plt.close()
    cycle = 100
    lw=2
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_axes([.18, .18, .7, .7])
    xrange = np.linspace(0,kmax, kmax)
    [im] = ax.plot(xrange,Ez_phy[int(kmax/2),:],linewidth=lw)
    [im2] = ax.plot(xrange,axion[int(kmax/2),:],linewidth=lw)
    im.set_color('blue')
    im2.set_color('orange')
    init(kmax)
    plt.ylim(-1, 1)
    plt.ylim(-0.1, 0.1)



for t in range(nsteps+1):
    pulse = get_source(t)
    
    if t == 0:
        axion_past = np.zeros([kmax, kmax],float)
    
    # update E
    Ex, Ey, Ez = Eupdate2d(Ex, Ey, Ez, Bx, By, Bz)
    # update physical fields
    Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
    # inject E source
    Ez_phy[jsource, isource] = Ez_phy[jsource, isource] + pulse
    # update hat fields
    Ex, Ey, Ez, Bx, By, Bz = phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion)
    
    # update B 
    Bx, By, Bz = Bupdate2d(Ex, Ey, Ez, Bx, By, Bz)
    
    if t == 600:
        # update physical fields
        Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
        # inject B source
        Bz_phy = inj_Bsource(Bz_phy)
        # update hat fields
        Ex, Ey, Ez, Bx, By, Bz = phy2hat(Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy, axion)
    
    # update physical fields
    Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
    # update A
    axion = Aupdate2d(axion, axion_past, Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy)
    
    if t == 0:
        axion_current = np.zeros([kmax,kmax],float)
    axion_past = axion_current
    axion_current = axion
    
    if t % cycle == 0:
        Ex_phy, Ey_phy, Ez_phy, Bx_phy, By_phy, Bz_phy = hat2phy(Ex, Ey, Ez, Bx, By, Bz, axion)
        if plot1d == False:
            graph(t, Ez_phy)
        if plot1d == True:
            im.set_ydata(Ez_phy[int(kmax/2),:]) # blue
            im2.set_ydata(axion[int(kmax/2),:]) # orange
            ax.set_title("frame time {}".format(t))
            plt.show()
            plt.pause(0.05)
print('done')
    

done


In [ ]:
@numba.jit(nopython=True)
def test(test2d):
    for j in range(kmax):
        for i in range(kmax):
            radius2 = (jsource-j)**2 + (isource-i)**2
            if radius2 >= sourceRadius**2:
                test2d[j,i] = 1
                
    return test2d
test2d = np.zeros([kmax,kmax])

test2d = test(test2d)
plt.clf()
plt.contourf(test2d)
plt.colorbar()
plt.show()

In [15]:
Ez_phy

array([[0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
       [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
       [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
       ...,
       [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        1.19431407e-228, 3.83055294e-229, 0.00000000e+000],
       [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        4.75410725e-229, 1.52591393e-229, 0.00000000e+000],
       [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
        0.00000000e+000, 0.00000000e+000, 0.00000000e+000]])

In [38]:
np.where(axion>10)

(array([215, 215, 215, ..., 991, 991, 991]),
 array([418, 459, 475, ..., 453, 454, 455]))